# 00 START 项目总架构：入口、出口和代码地图

你现在卡住的点不是某一行代码，而是还没有项目总地图。

这一本先回答 4 个顶层问题：

1. 这个项目的“入口”是什么？
2. 这个项目的“出口”是什么？
3. 每个文件夹和脚本分别负责哪一段？
4. 如果我要学习或修改某个功能，应该从哪里进去？

先看完这一本，再去看后面的 Qwen / tokenizer / SFT / LoRA / DPO，会清楚很多。

In [ ]:
from pathlib import Path
import json, os, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)

def exists(rel):
    return (PROJECT_ROOT / rel).exists()

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

def show_file(rel, start=1, end=80):
    lines = (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace").splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

## 1. 项目的总目标

一句话：

> 在本地 RTX 4060 Laptop GPU 上，用 Qwen2.5-0.5B-Instruct 跑通一个可解释的 LoRA SFT / DPO 后训练工程闭环。

这里有三个关键词：

- **Qwen base model**：原始底座模型，负责基本语言能力。
- **LoRA adapter**：训练出来的小补丁，不是完整大模型。
- **行为门禁**：固定 prompt 对比，判断模型是不是真的变好。

## 2. 从最高层看：项目输入和输出

```text
输入：
  1. base model: Qwen/Qwen2.5-0.5B-Instruct
  2. SFT 数据: system / user / assistant
  3. DPO 数据: prompt / chosen / rejected
  4. 配置参数: max_length, batch_size, lr, LoRA r 等
  5. 固定测试题: data/samples/custom_technical_prompts.jsonl

处理：
  1. infer.py 加载 Qwen 做推理
  2. train_sft_lora.py 用 SFT 数据训练 LoRA adapter
  3. compare_*.py 用固定 prompt 对比多个模型版本
  4. train_dpo.py 用偏好数据继续优化 adapter
  5. score_*.py 用规则辅助判断行为是否过关

输出：
  1. outputs/ 里的 LoRA adapter
  2. reports/ 里的实验报告和对比结果
  3. README / runbook / learning notebooks 里的项目解释
```

## 3. 文件夹地图：每个房间是干嘛的

In [ ]:
folders = [
    "scripts", "configs", "data/raw", "data/processed", "data/samples",
    "outputs", "reports", "notebooks", "project_learning_notebooks_zh", ".hf_cache"
]
for rel in folders:
    p = PROJECT_ROOT / rel
    print(f"{rel:34s}", "存在" if p.exists() else "缺失")

| 文件夹 | 入口是什么 | 出口是什么 | 意义 |
|---|---|---|---|
| `.hf_cache/` | Hugging Face 下载/缓存 | 本地模型和数据缓存 | 让 Qwen 可以离线加载 |
| `data/raw/` | 原始文本/原始数据 | 待清洗材料 | 原料区 |
| `data/processed/` | 清洗转换后的 JSONL | SFT/DPO 训练文件 | 模型真正吃的数据 |
| `data/samples/` | 固定 prompt | 对比评估输入 | 项目考试卷 |
| `scripts/` | 命令行参数 | 数据、adapter、报告 | 项目的机器按钮 |
| `configs/` | yaml 配置 | 训练参数 | 复现实验设置 |
| `outputs/` | 训练脚本输出 | LoRA adapter | 训练成果 |
| `reports/` | 对比/打分/人工总结 | md/jsonl/csv 报告 | 实验日记和面试证据 |
| `notebooks/` | 原始实践 notebook | smoke test / pipeline 学习 | 早期操作记录 |
| `project_learning_notebooks_zh/` | 学习者视角 | 解释型 notebook | 帮你读懂项目 |

## 4. 脚本地图：哪个脚本负责哪条链路

先记一个通用代码结构：

```text
parse_args / read_config
  -> load tokenizer / load data / load model
  -> train 或 generate 或 compare
  -> save adapter 或 write report
```

绝大多数脚本都按这个套路写。

In [ ]:
scripts = [
    "check_env.py",
    "infer.py",
    "prepare_data.py",
    "prepare_custom_technical_data.py",
    "train_sft_lora.py",
    "compare_three_outputs.py",
    "compare_four_outputs.py",
    "score_fixed_prompt_outputs.py",
    "prepare_tiny_dpo_data.py",
    "prepare_larger_dpo_v6_data.py",
    "train_dpo.py",
]
for name in scripts:
    p = PROJECT_ROOT / "scripts" / name
    print(f"scripts/{name:38s}", "OK" if p.exists() else "MISSING")

| 脚本 | 输入 | 输出 | 你要理解的意义 |
|---|---|---|---|
| `check_env.py` | 当前 Python 环境 | 环境打印 | 证明 CUDA / torch / transformers 可用 |
| `infer.py` | prompt + 可选 adapter | 模型回答 | Qwen 加载和推理入口 |
| `prepare_data.py` | Alpaca/demo 数据 | SFT JSONL | 公开数据转 Qwen chat 格式 |
| `prepare_custom_technical_data.py` | 自采集技术资料/seed | custom SFT JSONL | badcase 驱动的数据工程 |
| `train_sft_lora.py` | SFT train/eval JSONL | LoRA adapter | SFT 训练主入口 |
| `compare_three_outputs.py` | 固定 prompt + 两个 adapter | 三方回答 JSONL | base/public/custom 对比 |
| `compare_four_outputs.py` | 固定 prompt + 三个 adapter | 四方回答 JSONL | 加入 DPO 后的对比 |
| `score_fixed_prompt_outputs.py` | 对比 JSONL | 分数 JSONL/CSV/报告 | 透明规则行为门禁 |
| `prepare_*dpo*_data.py` | badcase/候选答案 | DPO preference JSONL | 构造 chosen/rejected |
| `train_dpo.py` | DPO JSONL + SFT adapter | DPO LoRA adapter | 偏好优化主入口 |

## 5. 三条主流程：你要先分清楚它们

### A. 推理流程

```text
用户 prompt
  -> scripts/infer.py
  -> AutoTokenizer.from_pretrained
  -> AutoModelForCausalLM.from_pretrained
  -> 可选 PeftModel.from_pretrained(adapter)
  -> model.generate
  -> 文本回答
```

这是最小闭环：模型能不能回答。

In [ ]:
find_lines("scripts/infer.py", "AutoTokenizer.from_pretrained", context=4)
find_lines("scripts/infer.py", "AutoModelForCausalLM.from_pretrained", context=4)
find_lines("scripts/infer.py", "model.generate", context=4)

### B. SFT 训练流程

```text
SFT JSONL: messages=[system,user,assistant]
  -> ChatSFTDataset 读取并 tokenizer
  -> labels: prompt 部分设 -100，只学 assistant
  -> 加载 Qwen base
  -> LoraConfig + get_peft_model
  -> Trainer.train
  -> outputs/sft_lora_xxx adapter
```

这是项目最重要的训练主线。

In [ ]:
find_lines("scripts/train_sft_lora.py", "class ChatSFTDataset", context=4)
find_lines("scripts/train_sft_lora.py", "labels =", context=6)
find_lines("scripts/train_sft_lora.py", "LoraConfig", context=8)
find_lines("scripts/train_sft_lora.py", "trainer.train", context=4)

### C. DPO 偏好流程

```text
DPO JSONL: prompt / chosen / rejected
  -> read_dpo_rows 读取偏好对
  -> 加载 policy model: base + SFT adapter，可训练
  -> 可选加载 reference model: base + SFT adapter，冻结
  -> DPOTrainer.train
  -> outputs/dpo_lora_xxx adapter
  -> 固定 prompt 行为门禁决定是否接受
```

In [ ]:
find_lines("scripts/train_dpo.py", "def read_dpo_rows", context=6)
find_lines("scripts/train_dpo.py", "def load_policy_model", context=6)
find_lines("scripts/train_dpo.py", "def load_reference_model", context=6)
find_lines("scripts/train_dpo.py", "DPOTrainer", context=8)

## 6. 如果你想修改项目，该从哪里改？

| 你想改什么 | 先看哪里 | 可能改哪里 |
|---|---|---|
| 换 base 模型 | `scripts/infer.py` / configs | `model_name` |
| 改推理温度或长度 | `scripts/infer.py` | `--temperature`, `--max_new_tokens` |
| 改 SFT 数据 | `data/processed/*.jsonl` 生成逻辑 | `prepare_data.py` / `prepare_custom_technical_data.py` |
| 改 SFT loss/token 逻辑 | `ChatSFTDataset.encode` | labels / max_length |
| 改 LoRA 大小 | `train_sft_lora.py` | `lora_r`, `lora_alpha`, `target_modules` |
| 改 DPO 数据 | `prepare_*dpo*_data.py` | chosen/rejected 构造 |
| 改 DPO 显存 | `configs/dpo_*.yaml` | max_length, batch_size, grad_accum |
| 改验收规则 | `score_fixed_prompt_outputs.py` | RUBRICS / forbidden phrases |

## 7. 当前项目最终出口是什么？

你要特别记住：项目最后不是“所有训练输出都接受”。

```text
推荐 checkpoint: outputs/sft_lora_qwen05b_custom_v3_from_v1_patch
最佳 DPO artifact: outputs/dpo_lora_qwen05b_naive_v6
```

推荐 checkpoint 和最佳 DPO artifact 不同，原因是：DPO v6 很有实验价值，但没有完全通过固定 prompt 行为门禁。

In [ ]:
for rel in ["outputs/sft_lora_qwen05b_custom_v3_from_v1_patch", "outputs/dpo_lora_qwen05b_naive_v6"]:
    print(rel, "OK" if exists(rel) else "MISSING")

## 8. 看完这一本后，你应该怎么继续？

现在你已经知道总结构了。后面按这个顺序学：

1. Qwen 怎么加载和推理。
2. tokenizer / chat template 怎么把文字变数字。
3. SFT 数据和 loss。
4. LoRA adapter 怎么贴到 Qwen。
5. adapter 怎么保存、加载、评估。
6. DPO preference 数据怎么训练。
7. 如何讲成面试工程闭环。